### Previews

In [1]:
!pip install -r requirements.txt

In [2]:
import pandas as pd
import seaborn as sns

from tqdm import tqdm
from google_play_scraper import Sort, reviews, app

%matplotlib inline
%config InlineBackend.figure_format='retina'

sns.set(style='whitegrid', palette='muted', font_scale=1.2)

#### Alguns App de Comida e Alimentacao no Brasil

1. iFodd ⇒ br.com.brainweb.ifood
2. Zé delivery ⇒ com.cerveceriamodelo.modelonow
3. McDonald's ⇒ com.mcdo.mcdonalds
4. Habibi's ⇒ habibs.alphacode.com.br
5. Uber Eats ⇒ com.ubercab.eats
6. Rappi ⇒ com.grability.rappi
7. Burker King ⇒ burgerking.com.br.appandroid
8. Aiqfome ⇒ com.vanuatu.aiqfome

In [3]:
apps_ids = [
    'br.com.brainweb.ifood',
    'com.cerveceriamodelo.modelonow',
    'com.mcdo.mcdonalds', 'habibs.alphacode.com.br',
    'com.ubercab.eats',
    'com.grability.rappi',
    'burgerking.com.br.appandroid',
    'com.vanuatu.aiqfome'
]

Dados de Cada Aplicativo

In [4]:
app_infos = []

for ap in tqdm(apps_ids):
    info = app(ap, lang='en', country='us')
    del info['comments']
    app_infos.append(info)

100%|██████████| 8/8 [00:03<00:00,  2.24it/s]


In [5]:
app_infos_df = pd.DataFrame(app_infos)
app_infos_df.head()

,title,description,descriptionHTML,summary,installs,minInstalls,realInstalls,score,ratings,reviews,...,contentRating,contentRatingDescription,adSupported,containsAds,released,lastUpdatedOn,updated,version,appId,url
0,iFood comida e mercado em casa,"Buy in markets, restaurants, pharmacies and pe...","Buy in markets, restaurants, pharmacies and pe...","Order food delivery, market, drinks, pharmacy ...","100,000,000+",100000000,128989852,4.567633,13252775,2701,...,Everyone,None,False,False,"Apr 25, 2012","Apr 7, 2025",1744056075,10.64.0,br.com.brainweb.ifood,https://play.google.com/store/apps/details?id=...
1,Zé Delivery de Bebidas,"With Zé Delivery, the drink reaches you in min...","With Zé Delivery, the drink reaches you in min...","Drink delivery: cold, in minutes and free deli...","10,000,000+",10000000,27168351,4.760870,1408365,313,...,Mature 17+,None,False,False,"Aug 19, 2016","Mar 17, 2025",1742226945,25.11.2,com.cerveceriamodelo.modelonow,https://play.google.com/store/apps/details?id=...
2,McDonald's Offers and Delivery,Enter the new McDonald's App and get exclusive...,Enter the new McDonald&#39;s App and get exclu...,"Enjoy esclusive discounts, promotions and coup...","50,000,000+",50000000,87711367,4.161327,1673235,3889,...,Everyone,None,True,True,"Mar 27, 2017","Apr 3, 2025",1743715331,4.13.1,com.mcdo.mcdonalds,https://play.google.com/store/apps/details?id=...
3,Habib's: Descontos e Delivery,Let's order a Habib's today?\r\nHere on the Ap...,Let&#39;s order a Habib&#39;s today?<br>Here o...,Want to end your hunger? Come to the Habib's App!,"10,000,000+",10000000,10274927,3.933333,336255,20,...,Everyone,None,False,False,None,"Feb 21, 2025",1740147543,4.1.44,habibs.alphacode.com.br,https://play.google.com/store/apps/details?id=...
4,Uber Eats: Food Delivery,Get food delivery to your doorstep from thousa...,Get food delivery to your doorstep from thousa...,"Food & Grocery Delivery App. Order Pizza, Sush...","100,000,000+",100000000,251784498,4.641414,6277173,354518,...,Everyone,None,True,True,"Feb 29, 2016","Apr 7, 2025",1744060255,Varies with device,com.ubercab.eats,https://play.google.com/store/apps/details?id=...


#### Apps Previews

Queremos:
* Conjunto de dados equilibrado — aproximadamente o mesmo número de avaliações para cada pontuação (1–5)
* Uma amostra representativa das avaliações para cada aplicativo

Podemos satisfazer o primeiro requisito usando a opção de pacote de scraping para filtrar a pontuação da avaliação. Para o segundo, classificaremos as avaliações por sua utilidade, que são as avaliações que o Google Play considera mais importantes.

In [6]:
app_reviews = []

for ap in tqdm(apps_ids):
    for score in list(range(1, 6)):
        for sort_order in [Sort.MOST_RELEVANT, Sort.NEWEST]:
            rvs, _ = reviews(
                ap,
                lang='pt',
                country='br',
                sort=sort_order,
                count= 200 if score == 3 else 100,
                filter_score_with=score
            )
            for r in rvs:
                r['sortOrder'] = 'most_relevant' if sort_order == Sort.MOST_RELEVANT else 'newest'
                r['appId'] = ap
            app_reviews.extend(rvs)

100%|██████████| 8/8 [00:26<00:00,  3.31s/it]


In [7]:
len(app_reviews)

9400

Salvando avaliações em um arquivo CSV

In [8]:
app_reviews_df = pd.DataFrame(app_reviews)

In [9]:
app_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,sortOrder,appId
0,c51ee54a-9298-41e8-88f2-7affcd7ec6c0,Hernan Fonseca,https://play-lh.googleusercontent.com/a-/ALV-U...,"Para mim, pedir pelo iFood só me traz uma cert...",1,13,10.64.0,2025-04-05 13:32:53,None,NaT,10.64.0,most_relevant,br.com.brainweb.ifood
1,4b835174-dc00-4df3-9311-959ceaba407b,Juliano,https://play-lh.googleusercontent.com/a-/ALV-U...,"Propaganda enganosa, descontos falsos, preços ...",1,46,10.64.0,2025-04-01 12:18:34,None,NaT,10.64.0,most_relevant,br.com.brainweb.ifood
2,00a76c33-93f8-4d82-ab4d-ef7ca9936add,Vênus Lilith,https://play-lh.googleusercontent.com/a-/ALV-U...,Não consigo utilizar a opção de pagamento na e...,1,56,10.62.0,2025-03-28 08:28:53,None,NaT,10.62.0,most_relevant,br.com.brainweb.ifood
3,eeda662b-a7c1-477f-831a-20a138cf09e0,Suelen Mantovani,https://play-lh.googleusercontent.com/a-/ALV-U...,"Tinha que ter a opção 0 estrela, aplicativo se...",1,5,10.64.0,2025-04-05 19:27:25,None,NaT,10.64.0,most_relevant,br.com.brainweb.ifood
4,b50547f5-ada0-4b18-b4e2-c382d1772288,Yanna Dias,https://play-lh.googleusercontent.com/a-/ALV-U...,O Ifood tem várias falhas no sistema! muitas v...,1,5,10.63.0,2025-04-04 20:25:39,None,NaT,10.63.0,most_relevant,br.com.brainweb.ifood


In [10]:
app_reviews_df.to_csv('data/preview.csv', index=None, header=True)